# Small instances, live

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/chris-dare-dev/k3-derived-isogenies/HEAD?labpath=research%2Fnotebooks%2F02-small-live-instances.ipynb)

Each cell runs a script of `experiments/` on an instance small enough for a free notebook server. Times measured on
one core of a desktop; a hosted notebook may be two to three times slower. The CM-method cells need PARI, through
`cypari2` (installed from a binary wheel on Linux).

In [ ]:
import json, time
from k3data import ensure_data, run, EXP
ensure_data()
def timed(*args):
    t = time.time(); out = run(*args); print(f"[{time.time() - t:.1f} s]"); return out

## 1. Enumerate every six-line surface over F₁₃ (about 2 s)

All configurations of six rational lines in general position, up to projective equivalence and relabelling, with
their character sums ([note 2](../docs/02-six-line-count.md)). The result is compared with the stored file.

In [ ]:
out = timed("sixlines.py", "enumerate", "--p", "13", "--out", "/dev/stdout", "--procs", "1")
rows = sorted(line for line in out.splitlines() if line.startswith("{"))
stored = sorted(line for line in (EXP / "sixlines_p13.jsonl").read_text().splitlines() if line.strip())
print(len(rows), "configurations; identical to the stored file:", rows == stored)

## 2. The abelian fourfold of one surface over F₁₇ (about 3 s)

For the configuration (2, 3, 4, 11), the curve of genus 5 and its Prym fourfold are built for all 30 ordered pairs of
lines, and the second exterior power of the fourfold's Frobenius quartic is compared with the polynomial of the
surface ([note 14](../docs/14-fourfold-of-a-six-line-surface.md)). "equal" or "twist" is a match.

In [ ]:
out = timed("sixlines_prym.py", "--check", "17", "2", "3", "4", "11")
print("\n".join(out.splitlines()[:8]))
from collections import Counter
print(Counter(line.split()[-1] for line in out.splitlines() if line.strip().startswith("pair")))

## 3. The CM method and the group action at q = 19 (about 10 s)

Theta constants at 300 digits, the class polynomial, its reduction modulo the prime above 19, the action of ideals on
the lattices, and the test that an ℓ-step is Frobenius in characteristic ℓ ([note 20](../docs/20-cm-method-and-the-action.md)).

In [ ]:
out = timed("sixlines_cmmethod.py", "action", "19", "2", "1", "1", "--digits", "300", "--primes", "3", "--all-subsets")
print("\n".join(l for l in out.splitlines() if l.startswith("[") or "label" in l))

Optional, about two minutes: the same at q = 23, where G is cyclic of order 6 (uncomment to run).

In [ ]:
# out = timed("sixlines_cmmethod.py", "action", "23", "2", "1", "1", "--digits", "700", "--primes", "3", "--all-subsets")
# print("\n".join(l for l in out.splitlines() if l.startswith("[") or "label" in l))

## 4. The self-test of the theta-constant formulas (about 1 s)

In [ ]:
out = timed("sixlines_cmmethod.py", "selftest", "--digits", "60")
print(out.splitlines()[-1])

## What does not fit here

Enumerating all surfaces at p = 41 takes about 17 minutes on 14 cores, the CM method on torsors of 12 to 16 surfaces
needs 2,000 to 3,000 digits and minutes to hours, and the Jacobian computations of notes 23 and 26 take over a minute
per curve. Their results are in the stored files, which the first notebook reads.